In [1]:
%env CUDA_VISIBLE_DEVICES=3
%env OMP_NUM_THREADS=16
%env HF_HOME=/mnt/LLM

import torch
import transformers

from async_reasoning.solver import AsyncReasoningSolver as Solver
from qwen35_gdn.qwen35_ar_patch import patch_qwen35_for_async_reasoning
from utils.answer_processing import find_last_valid_expression, check_equality_judge, check_equality_local_model

MODEL_NAME = "Qwen/Qwen3.5-27B"  # qwen3_5 model_type — hybrid GDN + full-attention
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype='auto', low_cpu_mem_usage=True, device_map=device, trust_remote_code=True,
)
model.eval()

# Install the AR-friendly Qwen3_5GatedDeltaNet.forward on every GDN layer.
# This is a no-op for vanilla generate; only AR's CombinedCacheView uses the new affine path.
patch = patch_qwen35_for_async_reasoning(model)
print(f"Patched {len(patch.originals)} GDN layers.")

system_tokens = [k for k in tokenizer.vocab.keys() if k.endswith("SYSTEM") or k.endswith("SYSTEM:")]
writer_forbidden_token_ix = [tokenizer.vocab[x] for x in ["</think>", "<|im_start|>", "<|endoftext|>"] + system_tokens]
thinker_forbidden_token_ix = [tokenizer.vocab[x] for x in ["<|im_start|>", "<|im_end|>", "<|endoftext|>"] + system_tokens]
end_of_think_token_ix = [tokenizer.vocab["</think>"]]

env: CUDA_VISIBLE_DEVICES=3
env: OMP_NUM_THREADS=16
env: HF_HOME=/mnt/LLM


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Patched 48 GDN layers.


In [2]:
import async_reasoning.prompting as _P
_orig_prompting_init = _P.AsyncReasoningPrompting.__init__

def _patched_prompting_init(self, problem):
    _orig_prompting_init(self, problem)
    self.mode_switching_prompt = (
        "<|im_start|>user\n"
        "You are an AI assistant that thinks and writes responses concurrently. "
        "Decide whether to write the next sentence or think more.\n"
    )
    self.mode_switching_question = (
        "<|im_end|>\n<|im_start|>user\n"
        "Looking at the partial thoughts and partial response above, what should "
        "you do next? Reply with a single word: write (keep writing the response) "
        "or think (pause to think more).\n"
        "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )
    self.yes_token = "write"   # "continue writing the response"
    self.no_token = "think"    # "pause and let the thinker work"

_P.AsyncReasoningPrompting.__init__ = _patched_prompting_init
print("Mode-switcher patched: write/think + direct framing (Sweep D winner).")

Mode-switcher patched: write/think + direct framing (Sweep D winner).


In [ ]:
problem = """Find the number of triples of nonnegative integers (a, b, c) satisfying a + b + c = 300 and a²b + a²c + b²a + b²c + c²a + c²b = 6,000,000."""
answer = "601"

solver = Solver(
    model,
    tokenizer,
    writer_forbidden_token_ix=writer_forbidden_token_ix,
    thinker_forbidden_token_ix=thinker_forbidden_token_ix,
    end_of_think_token_ix=end_of_think_token_ix,
    use_fast_kernel=False,
)
writer_output_str, thinker_output_str, token_times, eos_generated = solver.solve(
    problem, budget=2048, display_generation_in_real_time=True,
)

# State.thinker_and_writer

## Thinker mode


<think>
Let the given equations be
(1) $a + b + c = 300$
(2) $a^2b + a^2c + b^2a + b^2c + c^2a + c^2b = 6,000,000$

We are looking for the number of triples of nonnegative integers $(a, b, c)$ satisfying these equations.
Let $S_1 = a + b + c$, $S_2 = ab + bc + ca$, and $S_3 = abc$.
The elementary symmetric polynomials in $a, b, c$ are $e_1 = S_1$, $e_2 = S_2$, $e_3 = S_3$.
The given equations can be expressed in terms of these elementary symmetric polynomials.
Equation (1) is simply $e_1 = 300$.

Let's look at the expression in equation (2).
$E = a^2b + a^2c + b^2a + b^2c + c^2a + c^2b$.
We can factor this expression.
$E = ab(a+b) + bc(b+c) + ca(c+a)$.
Alternatively, we can relate it to $e_1, e_2, e_3$.
We know that $e_1 e_2 = (a+b+c)(ab+bc+ca) = a^2b + abc + a^2c + ab^2 + b^2c + abc + abc + bc^2 + c^2a$.
Let's expand this carefully.
$(a+b+c)(ab+bc+ca) = a(ab) + a(bc) + a(ca) + b(ab) + b(bc) + b(ca) + c(ab) + c(bc) + c(ca)$
$= a^2b + abc + a^2c + ab^2 + b^2c +

## Writer mode


</think>

To find the number of triples of nonnegative integers $(a, b, c)$ satisfying the given system of equations, let's analyze the given expressions.

The given equations are:
1. $a + b + c = 300$
2. $a^2b + a^2c + b^2a + b^2c + c^2a + c^2b = 6,000,000$

Let's express the second equation in terms of elementary symmetric polynomials.
Let $e_1 = a+b+c$, $e_2 = ab+bc+ca$, and $e_3 = abc$.
We know the identity:
$e_1 e_2 = (a+b+c)(ab+bc+ca

KeyboardInterrupt: 

: 